In [4]:
from datasets import load_dataset
from angle_emb import AnglE, AngleDataTokenizer
import os

import model_utils

In [16]:

os.environ["CUDA_VISIBLE_DEVICES"] = "0"  # Use GPU 0
os.environ["WANDB_DISABLED"] = "true"
# os.environ["WANDB_MODE"] = "offline"
# os.environ["WANDB_MODE"] = "disabled"

In [17]:


# 1. load pretrained model
angle = AnglE.from_pretrained('SeanLee97/angle-bert-base-uncased-nli-en-v1', max_length=128, pooling_strategy='cls').cuda()

# 2. load dataset
# `text1`, `text2`, and `label` are three required columns.
ds = load_dataset('mteb/stsbenchmark-sts')
ds = ds.map(lambda obj: {"text1": str(obj["sentence1"]), "text2": str(obj['sentence2']), "label": obj['score']})
ds = ds.select_columns(["text1", "text2", "label"])

# 3. transform data.
train_ds = ds['train'].shuffle().map(AngleDataTokenizer(angle.tokenizer, angle.max_length), num_proc=8)
valid_ds = ds['validation'].map(AngleDataTokenizer(angle.tokenizer, angle.max_length), num_proc=8)
test_ds = ds['test'].map(AngleDataTokenizer(angle.tokenizer, angle.max_length), num_proc=8)

# training_args = TrainingArguments(
#     output_dir='ckpts/sts-b',
#     num_train_epochs=5,
#     learning_rate=2e-5,
#     per_device_train_batch_size=32,
#     gradient_accumulation_steps=16,
#     warmup_steps=0,
#     save_steps=100,
#     eval_steps=1000,           # Note: The argument is 'eval_steps', not 'evaluation_strategy' with steps
#     logging_steps=100,
#     fp16=True,
#     report_to="none",          # This is the correct way to disable wandb logging
# )

# 4. fit.
angle.fit(
    train_ds=train_ds,
    valid_ds=valid_ds,
    output_dir='ckpts/sts-b',
    batch_size=32,
    epochs=5,
    learning_rate=2e-5,
    save_steps=100,
    eval_steps=1000,
    warmup_steps=0,
    gradient_accumulation_steps=16, #1,
    loss_kwargs={
        'cosine_w': 0.0,
        'ibn_w': 1.0,
        'cln_w': 1.0,
        'angle_w': 0.02,
        'cosine_tau': 20,
        'ibn_tau': 20,
        'angle_tau': 20
    },
    fp16=True,
    logging_steps=100,
)

# 5. evaluate
#corrcoef, accuracy = angle.evaluate(test_ds) #, device=angle.device)
score = model_utils.bi_encoder_evaluate(test_ds)
print('score:', score)

Map (num_proc=8):   0%|          | 0/5749 [00:00<?, ? examples/s]INFO:AnglE:Detect DatasetFormats.A: text1,text2,label
INFO:AnglE:Detect DatasetFormats.A: text1,text2,label
INFO:AnglE:Detect DatasetFormats.A: text1,text2,label
INFO:AnglE:Detect DatasetFormats.A: text1,text2,label
INFO:AnglE:Detect DatasetFormats.A: text1,text2,label
Map (num_proc=8):   2%|▏         | 124/5749 [00:00<00:22, 251.58 examples/s]INFO:AnglE:Detect DatasetFormats.A: text1,text2,label
INFO:AnglE:Detect DatasetFormats.A: text1,text2,label
INFO:AnglE:Detect DatasetFormats.A: text1,text2,label
Map (num_proc=8): 100%|██████████| 5749/5749 [00:01<00:00, 5475.08 examples/s] 
Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).
/home/shay/PycharmProjects/sensim/.venv/lib/python3.10/site-packages/angle_emb/angle.py:766: FutureWarning: `tokenizer` is deprecated and will be re

Step,Training Loss,Validation Loss


44it [00:02, 18.58it/s]                        


score: 0.8461541028246832
